# Core Objective: 
1. Fetching the BrightData Profile 
    a. Validating the data. 
2. Fetching the prompt 
    a. Adding the inputs from the raw files. 
3. Generating the response. 
    a. Manually, checking the response generated by the LLM.

In [ ]:
import time
import requests
import os

DATASET_ID = "gd_l1viktl72bvl7bjuj0"
API_KEY = os.getenv("BRIGHTDATA_API_KEY")

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

inputs = [
    {"url": "https://www.linkedin.com/in/reema-jain-81873553/"},
]

# Trigger the async collection
trigger_response = requests.post(
    "https://api.brightdata.com/datasets/v3/trigger",
    params={
        "dataset_id": DATASET_ID,
        "format": "json",
    },
    headers=headers,
    json=inputs,
)
trigger_response.raise_for_status()

snapshot_id = trigger_response.json()["snapshot_id"]
print("Snapshot ID:", snapshot_id)

# Step 2: Poll progress until ready
while True:
    progress_response = requests.get(
        f"https://api.brightdata.com/datasets/v3/progress/{snapshot_id}",
        headers={"Authorization": f"Bearer {API_KEY}"},
    )
    progress_response.raise_for_status()

    status = progress_response.json().get("status")
    print("Status:", status)

    if status == "ready":
        break
    if status == "failed":
        raise Exception(f"Collection failed: {progress_response.text}")

    time.sleep(10)

# Step 3: Download results
results_response = requests.get(
    f"https://api.brightdata.com/datasets/v3/snapshot/{snapshot_id}",
    params={"format": "json"},
    headers={"Authorization": f"Bearer {API_KEY}"},
)
results_response.raise_for_status()

Snapshot ID: sd_mtcp13p21t6suz55bc
Status: running
Status: running
Status: ready
Collected 1 profiles
[{'id': 'reema-jain-81873553', 'name': 'Reema Jain', 'city': 'Thane, Maharashtra, India', 'country_code': 'IN', 'position': None, 'about': 'Driving Digital Excellence Through Strategic Talent Acquisition As a Senior Talent…', 'posts': [{'title': 'Hiring for Developers', 'attribution': 'Job description- Senior Java Developer The Senior Java Developer role is to design, develop and support OLM’s web-based…', 'img': 'https://media.licdn.com/dms/image/v2/C5112AQG6y0AgZBPOXA/article-cover_image-shrink_600_2000/article-cover_image-shrink_600_2000/0/1520116067058?e=2147483647&v=beta&t=3BVkaIc5-x6FPbmWDKPBlf-Y_lMJ9Km1uks7kiRMbt8', 'link': 'https://www.linkedin.com/pulse/hiring-developers-reema-ashi-jain', 'created_at': '2016-07-19T00:00:00.000Z', 'interaction': '- 2 Comments', 'id': '6161133226749464576'}, {'title': 'Hiring for IOS Developers', 'attribution': 'Job Description: IOS Developers C

In [60]:
def process_enrich_data(user_data, indent=0, enrich_string: str = "") -> str:
    """
    Recurisively prints a nested dictionary in a tree-like format.
    Return:
        enrich_str: The enriched string representation of the nested dictionary.
    """
    if not isinstance(user_data, dict):
        enrich_string += (" " * indent + str(user_data)) + "\n"
        return enrich_string

    for key, value in user_data.items():
        enrich_string += (" " * indent + str(key) + ":") + "\n"

        if isinstance(value, dict):
            process_enrich_data(value, indent + 4, enrich_string)
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    process_enrich_data(item, indent + 4, enrich_string)
                else:
                    enrich_string += (" " * (indent + 4) + str(item)) + "\n"
        else:
            enrich_string += (" " * (indent + 4) + str(value)) + "\n"

    return enrich_string


results = results_response.json()
# print(f"Collected {len(results)} profiles")
print(process_enrich_data(results[0]))
# Print the response properly.
# Make a function to print things properly.

id:
    reema-jain-81873553
name:
    Reema Jain
city:
    Thane, Maharashtra, India
country_code:
    IN
position:
    None
about:
    Driving Digital Excellence Through Strategic Talent Acquisition As a Senior Talent…
posts:
current_company:
experience:
    None
url:
    https://dk.linkedin.com/in/Reema-jain-81873553
people_also_viewed:
educations_details:
    Narsee Monjee Institute of Management Studies
education:
recommendations_count:
    3
avatar:
    https://media.licdn.com/dms/image/v2/D4D03AQHlml8tfOn9LA/profile-displayphoto-shrink_200_200/B4DZdEg4XxG8AY-/0/1749201160386?e=2147483647&v=beta&t=q0iJxoT-w-NmcaSbLaSzbBM-G2pnkLQYZCAsux8RkpQ
languages:
certifications:
recommendations:
    Sudhir Shinde “Reema is a perfect example of energy, coupled with passion for Talent Acquisition. She was leading the Campus initiative for NielsenIQ and was instrumental in sourcing the entry level talent. She is extremely energetic and a treat to work with. I am sure she will go places with this

In [ ]:
import yaml

outreach_prompt_path = "/home/gautamw_7/Desktop/Projects/job-search-assistant/prompts/outreach_message.yaml"
with open(outreach_prompt_path, "r") as f:
    outreach_prompt = yaml.safe_load(f)

print(outreach_prompt)

system_prompt = outreach_prompt["system_prompt"]
user_prompt_template = outreach_prompt["user_prompt_template"]
print(user_prompt_template)

=== USER RESUME ===
{user_resume}

=== USER COMMUNICATION STYLE ===
{user_communication_style}

=== MESSAGE PERSONALIZATION ===
{message_personalization}

=== RECIPIENT ENRICHED DATA ===
{recipient_enrich_data}

=== USER OUTREACH NOTES ===
{user_notes}



In [69]:
import PyPDF2

resume_path = "/home/gautamw_7/Desktop/Projects/job-search-assistant/data/Gautam_Warvatkar_Resume.pdf"
with open(resume_path, "rb") as f:
    pdf_reader = PyPDF2.PdfReader(f)
    resume_text = ""
    for page in pdf_reader.pages:
        resume_text += page.extract_text()

print(resume_text)
input_data = {}
input_data["user_resume"] = resume_text

Gautam Warvatkar
gautam.warvatkar@gmail.com /ne+91 9987171860 /gtbGautamw7 Gautam Warvatkar
Technical Skills
Languages: Python, SQL
Machine Learning: XGBoost, Scikit-learn, TensorFlow/Keras, CNNs, feature engineering, predictive modeling
AI / NLP: RAG, Sentence Transformers, Gemini, embeddings, LLM re-ranking
Data Engineering: Pandas, NumPy, ETL pipelines, FAISS, Qdrant, MongoDB
AWS:S3, SageMaker Pipelines, Redshift, AWS Glue, boto3, Lambda, KMS
Tools:Git, Jira, Confluence, Power Automate, Smartsheet
Experience
Data Scientist / Engineer Intern Feb 2026 – Aug 2026
Digital Productivity & Innovation, Syngenta Pune, India
•Supported transfer decisions across 11 distribution centers and runs covering up to $75M/day in
product value by engineering an explainable multi-gate Go/Caution/Stop decision framework using sales orders,
inventory, historical consumption, and demand signals, targeting an official $6M/year SOOR reduction opportunity.
•Exposed 85 transfer-decision flips, including 65 Go/

In [70]:
user_communication_style_yaml_path = "/home/gautamw_7/Desktop/Projects/job-search-assistant/prompts/user_communication.yaml"

with open(user_communication_style_yaml_path, "r") as f:
    user_communication_style = yaml.safe_load(f)

input_data["user_communication_style"] = user_communication_style["default"]

In [71]:
user_preferences_yaml_path = "/home/gautamw_7/Desktop/Projects/job-search-assistant/prompts/message_personalization.yaml"
with open(user_preferences_yaml_path, "r") as f:
    message_personalization = yaml.safe_load(f)

input_data["message_personalization"] = message_personalization["default"]

In [72]:
input_data["recipient_enrich_data"] = process_enrich_data(results[0])
input_data["user_notes"] = (
    "She is a Senior TA. just a friendly, message asking her to keep me in mind for future syngenta role"
)

In [73]:
input_data.keys()

dict_keys(['user_resume', 'user_communication_style', 'message_personalization', 'recipient_enrich_data', 'user_notes'])

In [ ]:
# format user_prompt_template with input_data
user_prompt_template = user_prompt_template.format(**input_data)

prompt = system_prompt + "\n" + user_prompt_template
print(prompt)

KeyError: 'user_resume'

In [ ]:
# Pass

{'resume': 'Gautam Warvatkar\ngautam.warvatkar@gmail.com /ne+91 9987171860 /gtbGautamw7 Gautam Warvatkar\nTechnical Skills\nLanguages: Python, SQL\nMachine Learning: XGBoost, Scikit-learn, TensorFlow/Keras, CNNs, feature engineering, predictive modeling\nAI / NLP: RAG, Sentence Transformers, Gemini, embeddings, LLM re-ranking\nData Engineering: Pandas, NumPy, ETL pipelines, FAISS, Qdrant, MongoDB\nAWS:S3, SageMaker Pipelines, Redshift, AWS Glue, boto3, Lambda, KMS\nTools:Git, Jira, Confluence, Power Automate, Smartsheet\nExperience\nData Scientist / Engineer Intern Feb 2026 – Aug 2026\nDigital Productivity & Innovation, Syngenta Pune, India\n•Supported transfer decisions across 11 distribution centers and runs covering up to $75M/day in\nproduct value by engineering an explainable multi-gate Go/Caution/Stop decision framework using sales orders,\ninventory, historical consumption, and demand signals, targeting an official $6M/year SOOR reduction opportunity.\n•Exposed 85 transfer-decis